# Load test: does a Qdrant server remove the retrieval bottleneck?

Runs on **Kaggle, GPU T4 x2, Internet ON**, with the `civil-code-corpus` dataset added. Same setup as `kaggle_load_test.ipynb` (FastAPI app → Qwen3-8B-AWQ on vLLM, Locust on the same machine), with the retrieval changes from `docs/decisions.md` ("Retrieval under load").

The first load test measured **2.45 s of retrieval per request at 50 users** (38% of request time). This notebook runs **two 50-user tests, 5 min each**, with the new code:

- **A. local Qdrant**: only the code change (same-language twin fetched by ID instead of a filtered scan)
- **B. Qdrant server**: the code change plus a Qdrant server with payload indexes (`QDRANT_URL`)

Each run gets a fresh API process, so its `/metrics` histogram covers that run only. The baseline is the committed `reports/locust_u50*` and `reports/load_metrics.txt`.

Output: `reports/locust_local_u50*`, `reports/locust_server_u50*`, `reports/load_metrics_local.txt`, `reports/load_metrics_server.txt` and `reports/load_test_qdrant.md` (comparison table).

Long cells write to log files; if the display freezes, reload the page (F5). The work continues. Budget: about 40 GPU-minutes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code and corpus

In [ ]:
!git clone -q https://github.com/ZeinabMahfouz/The-Egyptian-Civil-Code-RAG.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1
!mkdir -p data/interim eval_out && cp "$(find /kaggle/input -name civil_code.json | head -1)" data/interim/
# The two md5 values must match
!md5sum data/interim/civil_code.json
!grep -A3 "path: data/interim/civil_code.json" dvc.lock | grep "md5:"

## 2. vLLM in its own environment

In [ ]:
!pip install -q uv
!uv venv /kaggle/vllm-env --python 3.11
!uv pip install -q --python /kaggle/vllm-env/bin/python vllm wrapt
!/kaggle/vllm-env/bin/python -c "import vllm, torch; print('vllm', vllm.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
import os, signal, subprocess, time, urllib.request

def start(cmd, log_path, env=None):
    return subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT,
                            env={**os.environ, **(env or {})}, start_new_session=True)

def wait_ready(url, log_path, minutes=30):
    for i in range(minutes * 4):
        try:
            urllib.request.urlopen(url, timeout=5); print(f"ready after ~{i*15}s"); return True
        except Exception:
            time.sleep(15)
    print("NOT ready -- log:"); os.system(f"tail -n 30 {log_path}"); return False

def stop(proc):
    os.killpg(proc.pid, signal.SIGTERM); proc.wait(timeout=60); time.sleep(3)

## 3. API environment (FastAPI app + Locust)

In [ ]:
!pip install -q fastapi uvicorn "qdrant-client==1.19.1" sentence-transformers pyyaml "openai==1.109.1" \
    langfuse==4.15.6 prometheus-client==0.26.0 locust
!pip install -q -e . --no-deps

## 4. Build the index (same chunking and embedding as `dvc repro`, on GPU)

In [ ]:
!python scripts/chunk_corpus.py --corpus data/interim/civil_code.json --out data/interim/chunks.json --params params.yaml
!python scripts/embed_and_index.py --chunks data/interim/chunks.json --params params.yaml --docs-dir data/documents 2>&1 | grep -v "Batches:" | tail -n 4

## 5. Qdrant server, seeded from the index just built

In [ ]:
!curl -sSL -o /kaggle/working/qdrant.tgz https://github.com/qdrant/qdrant/releases/download/v1.19.0/qdrant-x86_64-unknown-linux-musl.tar.gz
!tar xzf /kaggle/working/qdrant.tgz -C /kaggle/working && /kaggle/working/qdrant --version
qdrant = start(["/kaggle/working/qdrant"], "/kaggle/working/qdrant.log",
               env={"QDRANT__STORAGE__STORAGE_PATH": "/kaggle/working/qdrant_server_storage",
                    "QDRANT__TELEMETRY_DISABLED": "true"})
wait_ready("http://localhost:6333/", "/kaggle/working/qdrant.log", minutes=2)
# Copies data/processed/qdrant_storage into the server, with payload indexes
!python -m egyptian_civil_code_rag.qdrant_seed --url http://localhost:6333

## 6. Start vLLM (AWQ)

In [ ]:
# 0.6 GPU memory: AWQ weights are only ~2.9 GiB per GPU; the rest is KV cache for concurrent requests,
# with room left for the API's BGE-M3 on cuda:0.
vllm = start(["/kaggle/vllm-env/bin/vllm", "serve", "Qwen/Qwen3-8B-AWQ",
              "--tensor-parallel-size", "2", "--dtype", "half", "--max-model-len", "8192",
              "--gpu-memory-utilization", "0.6", "--port", "8000"],
             "/kaggle/working/vllm.log", env={"PYTHONPATH": ""})
wait_ready("http://localhost:8000/v1/models", "/kaggle/working/vllm.log")

## 7. Run A: new code, local Qdrant (50 users, 5 min)

In [ ]:
VLLM_ENV = {"VLLM_BASE_URL": "http://localhost:8000/v1", "GEN_MODEL": "Qwen/Qwen3-8B-AWQ"}
api = start(["uvicorn", "egyptian_civil_code_rag.api:app", "--host", "0.0.0.0", "--port", "8001"],
            "/kaggle/working/api_local.log", env=VLLM_ENV)
wait_ready("http://localhost:8001/health", "/kaggle/working/api_local.log", minutes=10)
!curl -s http://localhost:8001/health; echo
!locust -f load_test/locustfile.py --host http://localhost:8001 --headless -u 50 -r 5 -t 5m \
    --csv reports/locust_local_u50 --html reports/locust_local_u50.html --only-summary > eval_out/locust_local_u50.log 2>&1; tail -n 8 eval_out/locust_local_u50.log
!curl -s http://localhost:8001/metrics | grep "^rag_" > reports/load_metrics_local.txt
stop(api)

## 8. Run B: new code, Qdrant server (50 users, 5 min)

In [ ]:
api = start(["uvicorn", "egyptian_civil_code_rag.api:app", "--host", "0.0.0.0", "--port", "8001"],
            "/kaggle/working/api_server.log", env={**VLLM_ENV, "QDRANT_URL": "http://localhost:6333"})
wait_ready("http://localhost:8001/health", "/kaggle/working/api_server.log", minutes=10)
# documents_indexed must be 2197, served by the Qdrant server
!curl -s http://localhost:8001/health; echo
!curl -s -X POST http://localhost:8001/ask -H "Content-Type: application/json" -d '{"question": "Is Article 400 still in force?"}'; echo
!locust -f load_test/locustfile.py --host http://localhost:8001 --headless -u 50 -r 5 -t 5m \
    --csv reports/locust_server_u50 --html reports/locust_server_u50.html --only-summary > eval_out/locust_server_u50.log 2>&1; tail -n 8 eval_out/locust_server_u50.log
!curl -s http://localhost:8001/metrics | grep "^rag_" > reports/load_metrics_server.txt
stop(api)

## 9. Compare with the first load test

In [ ]:
# Row 1 is the committed first run (old code, local Qdrant; its metrics also include the 1-user run)
!python scripts/locust_summary.py reports/locust_u50 reports/locust_local_u50 reports/locust_server_u50 \
    --metrics reports/load_metrics.txt reports/load_metrics_local.txt reports/load_metrics_server.txt \
    --out reports/load_test_qdrant.md

## 10. Download

In [ ]:
!zip -qr /kaggle/working/load_test_qdrant_results.zip reports/locust_local_u50* reports/locust_server_u50* \
    reports/load_metrics_local.txt reports/load_metrics_server.txt reports/load_test_qdrant.md eval_out/locust_*_u50.log
!cp /kaggle/working/api_*.log /kaggle/working/qdrant.log eval_out/ && zip -qr /kaggle/working/load_test_qdrant_results.zip eval_out/api_*.log eval_out/qdrant.log
%cd /kaggle/working
from IPython.display import FileLink
display(FileLink("load_test_qdrant_results.zip"))